In [1]:
!pip install torch transformers datasets accelerate bitsandbytes trl peft wandb sentencepiece protobuf huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 427.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 230.5 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nv

In [2]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainerCallback

2025-04-22 06:34:21.249984: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745303661.458786      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745303661.517912      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Load Dataset
print("Loading the OASST1 dataset...")
dataset = load_dataset("OpenAssistant/oasst1")
print(f"Dataset loaded successfully. Keys: {dataset.keys()}")

# Filter for English conversations
dataset = dataset.filter(lambda x: x['lang'] == 'en')

# Build conversation threads
print("Building conversation threads...")
message_children = {}
for example in dataset['train']:
    if example["parent_id"] is not None:
        message_children[example["parent_id"]] = example

# Format conversations
def format_conversation(example):
    """Format the conversation for instruction fine-tuning"""
    # Only process root messages (start of conversations)
    if example["role"] == "prompter" and example["parent_id"] is None:
        conversation = []
        current_msg = example
        conversation.append(("Human", current_msg["text"]))
        
        # Follow the conversation thread
        current_id = current_msg["message_id"]
        while current_id in message_children:
            # Get the next message in conversation
            next_msg = message_children[current_id]
            if next_msg["role"] == "assistant":
                conversation.append(("Assistant", next_msg["text"]))
            elif next_msg["role"] == "prompter":
                conversation.append(("Human", next_msg["text"]))
            current_id = next_msg["message_id"]
            
        if len(conversation) >= 2:  # At least one exchange (human->assistant)
            formatted_text = ""
            for speaker, text in conversation:
                formatted_text += f"{speaker}: {text}\n\n"
            return {"text": formatted_text.strip()}
    return {"text": None}

# Build formatted dataset
print("\nFormatting conversations...")
processed_dataset = []
for example in dataset['train']:
    result = format_conversation(example)
    if result["text"] is not None:
        processed_dataset.append(result)
    if len(processed_dataset) % 100 == 0 and len(processed_dataset) > 0:
        print(f"Found {len(processed_dataset)} valid conversations")

print(f"Final dataset size: {len(processed_dataset)} conversations")

# Convert to Dataset format
train_dataset = Dataset.from_list(processed_dataset)

# Sample display
print("\nSample conversation:")
print(train_dataset[0]['text'])

# Load and Configure Tokenizer
print("Loading and configuring the tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2", trust_remote_code=True)

# Fix: Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Padding token set to EOS token")

# Configure and Load Model
print("Configuring and loading the model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)
print("Model and dataset loaded successfully")

# Print model architecture to identify target modules
print("Model architecture for LoRA configuration:")
print(model)


Loading the OASST1 dataset...


README.md:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

(…)-00000-of-00001-b42a775f407cee45.parquet:   0%|          | 0.00/39.5M [00:00<?, ?B/s]

(…)-00000-of-00001-134b8fd0c89408b6.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and dataset loaded successfully
Model architecture for LoRA configuration:
PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear4bit(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear4bit(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear4bit(in_features=2560, out_features=2560, bias=True)
          (dense): Linear4bit(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear4bit(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear4bit(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbed

In [ ]:
# Tokenize Dataset 
print("Tokenizing the dataset...")
def tokenize_function(examples):
    # Add truncation and padding settings
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=1024,
        return_tensors=None  # Set this explicitly
    )

# Apply tokenization to the formatted dataset created 
tokenized_datasets = train_dataset.map(tokenize_function, batched=True)
print(f"Dataset tokenized successfully. Total examples: {len(tokenized_datasets)}")

# Print sample tokenized example
if len(tokenized_datasets) > 0:
    print("\nSample tokenized data:")
    # Get first example safely
    try:
        sample = tokenized_datasets[0]
        # Check if it's a dictionary
        if isinstance(sample, dict) and "input_ids" in sample:
            print(f"Input IDs length: {len(sample['input_ids'])}")
            print(f"Attention mask length: {len(sample['attention_mask'])}")
            
            # Print the data structure type
            print(f"Dataset element type: {type(sample)}")
            print(f"Keys in first example: {list(sample.keys())}")
            
            # Confirm all examples have consistent length - safely
            try:
                # Get just a few examples
                first_examples = [tokenized_datasets[i] for i in range(min(10, len(tokenized_datasets)))]
                lengths = [len(ex["input_ids"]) for ex in first_examples if isinstance(ex, dict) and "input_ids" in ex]
                print(f"Sample example lengths: {lengths}")
            except Exception as e:
                print(f"Error checking lengths: {e}")
        else:
            print(f"Unexpected format. Sample: {str(sample)[:200]}...")
    except Exception as e:
        print(f"Error accessing sample: {e}")

# Verify dataset is ready for training
print(f"Dataset column names: {tokenized_datasets.column_names}")
print(f"Dataset format: {tokenized_datasets.format}")

# Let's try to properly format the dataset for the trainer
print("Reformatting dataset for training...")
tokenized_datasets = tokenized_datasets.remove_columns(["text"])  # Remove original text
tokenized_datasets.set_format("torch")  # Convert to PyTorch tensors
print("Dataset reformatted successfully.")


Tokenizing the dataset...


Map:   0%|          | 0/3482 [00:00<?, ? examples/s]

Dataset tokenized successfully. Total examples: 3482

Sample tokenized data:
Input IDs length: 1024
Attention mask length: 1024
Dataset element type: <class 'dict'>
Keys in first example: ['text', 'input_ids', 'attention_mask']
Sample example lengths: [1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024]
Dataset column names: ['text', 'input_ids', 'attention_mask']
Dataset format: {'type': None, 'format_kwargs': {}, 'columns': ['text', 'input_ids', 'attention_mask'], 'output_all_columns': False}
Reformatting dataset for training...
Dataset reformatted successfully.


In [ ]:
from transformers import TrainerCallback, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os
import datasets

# Force PyTorch to use a single GPU for now
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only the first GPU

# Note: We assume that dataset and tokenized_datasets are already prepared in previous cells 
print("Using tokenized dataset from previous cell")

# Load tokenizer again to ensure it's available in this cell
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

# Print dataset information
print(f"Dataset info - format: {tokenized_datasets.format}, columns: {tokenized_datasets.column_names}")

# Configure and Load Model
print("Configuring and loading the model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map={"": 0},  # Force all modules to the first GPU
)
print("Model loaded successfully")

# Prepare model for training
print("Preparing model for k-bit training...")
model.config.use_cache = False  # Disable caching
model = prepare_model_for_kbit_training(model)

# Configure LoRA with correct modules
print("Configuring LoRA...")
target_modules = ["q_proj", "k_proj", "v_proj", "dense"]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
print("LoRA configuration complete")

# Set up more efficient training arguments
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="phi2-oasst1-qlora",
    per_device_train_batch_size=1,  # Use batch size 1 to avoid shape issues
    gradient_accumulation_steps=16,  # Increased to compensate for smaller batch
    max_steps=500,  # Set to 500 steps
    max_grad_norm=0.3,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    logging_steps=1,
    save_strategy="steps",
    save_steps=100,  # Save every 100 steps
    disable_tqdm=False,
    report_to="none",
    fp16=True,  # Use fp16 precision
    dataloader_pin_memory=False,  # Try disabling pin memory
    dataloader_drop_last=True,  # Drop last incomplete batch
)
print("Training arguments set up successfully")

# Add this in the train loop if needed
class PrintCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0:  # Every 10 steps
            print(f"Step {state.global_step}: Loss = {state.log_history[-1]['loss'] if state.log_history else 'N/A'}")

# Initialize Trainer
print("Initializing SFTTrainer...")

# Try a simplified approach using basic Trainer 
try:
    from transformers import DataCollatorForLanguageModeling
    
    # Create a data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # We're not doing masked language modeling
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=tokenized_datasets,
        args=training_args,
        callbacks=[PrintCallback()],
        data_collator=data_collator,  # Use the data collator
    )
    print("SFTTrainer initialized successfully")
except Exception as e:
    print(f"Error initializing SFTTrainer: {e}")
    # Fall back to basic Trainer
    print("Falling back to basic Trainer...")
    trainer = Trainer(
        model=model,
        train_dataset=tokenized_datasets,
        args=training_args,
        callbacks=[PrintCallback()],
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    print("Basic Trainer initialized successfully")

# Train the Model for fixed steps
print("Starting the training process...")
try:
    import time
    start_time = time.time()
    trainer.train()
    print(f"Training completed in {time.time() - start_time:.2f} seconds")
except Exception as e:
    print(f"Training error: {e}")
    import traceback
    traceback.print_exc()
    # Try to save even if there was an error
    try:
        print("Attempting to save the partially trained model...")
        trainer.save_model("phi2-oasst1-qlora")
        print("Partial model saved successfully")
    except Exception as save_error:
        print(f"Could not save the model: {save_error}")

# Save the Model
print("Saving the model...")
try:
    trainer.save_model("phi2-oasst1-qlora")
    print("Model saved successfully to 'phi2-oasst1-qlora' directory")
except Exception as e:
    print(f"Error saving model: {e}")
    # Try alternative saving method
    try:
        model.save_pretrained("phi2-oasst1-qlora")
        print("Model saved using alternative method")
    except Exception as e2:
        print(f"All saving methods failed: {e2}")

print("Training complete!")


Using tokenized dataset from previous cell
Loading tokenizer...
Tokenizer loaded
Dataset info - format: {'type': 'torch', 'format_kwargs': {}, 'columns': ['input_ids', 'attention_mask'], 'output_all_columns': False}, columns: ['input_ids', 'attention_mask']
Configuring and loading the model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully
Preparing model for k-bit training...
Configuring LoRA...
LoRA configuration complete
Setting up training arguments...
Training arguments set up successfully
Initializing SFTTrainer...


Truncating train dataset:   0%|          | 0/3482 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


...

Step 210: Loss = 1.3797
Step 220: Loss = 1.5798
Step 230: Loss = 1.4682
Step 240: Loss = 1.9687
Step 250: Loss = 1.5841
Step 260: Loss = 1.6988
Step 270: Loss = 1.6397
Step 280: Loss = 1.3719
Step 290: Loss = 1.5816
Step 300: Loss = 1.403


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 310: Loss = 1.7005
Step 320: Loss = 1.5958
Step 330: Loss = 1.815
Step 340: Loss = 1.5892
Step 350: Loss = 1.4256
Step 360: Loss = 1.3797
Step 370: Loss = 1.4903
Step 380: Loss = 1.7199
Step 390: Loss = 1.6139
Step 400: Loss = 1.5575


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 410: Loss = 1.5726
Step 420: Loss = 1.5041
Step 430: Loss = 2.0053
Step 440: Loss = 1.5227
Step 450: Loss = 2.0383
Step 460: Loss = 1.6274
Step 470: Loss = 1.3634
Step 480: Loss = 1.3245
Step 490: Loss = 1.3591
Step 500: Loss = 1.5619
Training completed in 29877.98 seconds
Saving the model...
Model saved successfully to 'phi2-oasst1-qlora' directory
Training complete!


In [1]:
# Code to load the model
try:
    # Most robust approach - manual LoRA application
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    import torch
    import os

    print("Loading model components...")
    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2", trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load the base model
    base_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/phi-2", 
        trust_remote_code=True,
        device_map="auto"
    )
    
    # Configure LoRA (same config as training)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "dense"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    
    # Apply LoRA to the model
    model = get_peft_model(base_model, lora_config)
    
    # First check for safetensors format (newer format)
    adapter_path = "phi2-oasst1-qlora/adapter_model.safetensors"
    if not os.path.exists(adapter_path):
        # Fall back to bin format if safetensors doesn't exist
        adapter_path = "phi2-oasst1-qlora/adapter_model.bin"
    
    if os.path.exists(adapter_path):
        print(f"Loading adapter weights from {adapter_path}")
        # For safetensors format
        if adapter_path.endswith(".safetensors"):
            from safetensors.torch import load_file
            state_dict = load_file(adapter_path)
        else:
            # For bin format
            state_dict = torch.load(adapter_path, map_location="cpu")
            
        # Filter out incompatible keys
        filtered_dict = {k: v for k, v in state_dict.items() if k in model.state_dict()}
        model.load_state_dict(filtered_dict, strict=False)
        print(f"Loaded {len(filtered_dict)}/{len(state_dict)} parameters successfully")
    else:
        print(f"Error: No adapter weights found. Tried both safetensors and bin formats.")
        
    print("Model loaded successfully")

except Exception as e:
    print(f"Error during model loading: {e}")
    import traceback
    traceback.print_exc()
    print("\nPlease check that the model was trained and saved correctly.")

2025-04-22 16:08:09.472340: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745338089.660913     128 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745338089.714080     128 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading model components...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading adapter weights from phi2-oasst1-qlora/adapter_model.safetensors
Loaded 0/256 parameters successfully
Model loaded successfully


In [ ]:
# Code to test the model loaded 
try:
    # This script assumes model, tokenizer, and device are available in the current session
    
    # Test with a sample prompt
    sample_text = "Human:Write a motivational paragraph for someone preparing for an exam\n\nAssistant:"
    print(f"Tokenizing prompt: '{sample_text}'")
    inputs = tokenizer(sample_text, return_tensors="pt")
    
    # Move inputs to the correct device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    print(f"Generating response...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nModel Response: {response}")
    
    # Try another example
    sample_text2 = "Human: Explain the concept of quantum computing in simple terms\n\nAssistant:"
    print(f"\nTokenizing prompt: '{sample_text2}'")
    inputs = tokenizer(sample_text2, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    print(f"Generating response...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nModel Response: {response}")

except Exception as e:
    print(f"Error during model testing: {e}")
    import traceback
    traceback.print_exc()
    print("\nPlease make sure you've run the previous cell first to load the model.")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Tokenizing prompt: 'Human:Write a motivational paragraph for someone preparing for an exam

Assistant:'
Generating response...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Model Response: Human:Write a motivational paragraph for someone preparing for an exam

Assistant: You have worked hard to prepare for this exam, and you have the knowledge and skills to succeed. You have overcome many challenges and obstacles, and you have learned from your mistakes and improved your performance. You have a clear goal and a positive attitude, and you are ready to face any question that comes your way. You have nothing to fear, and everything to gain. You are capable, confident, and courageous, and you can achieve anything you set your mind to. Go and ace that exam, and celebrate your success!


Tokenizing prompt: 'Human: Explain the concept of quantum computing in simple terms

Assistant:'
Generating response...

Model Response: Human: Explain the concept of quantum computing in simple terms

Assistant: Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations. Unlike classical computers, which use bits that can be

In [ ]:
# Code to test the model loaded 
try:
    # This script assumes model, tokenizer, and device are available in the current session
    
    # Test with a sample prompt
    sample_text = "Human:Explain how a neural network works like you’re teaching it to someone who’s never coded\n\nAssistant:"
    print(f"Tokenizing prompt: '{sample_text}'")
    inputs = tokenizer(sample_text, return_tensors="pt")
    
    # Move inputs to the correct device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    print(f"Generating response...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nModel Response: {response}")
    
    # Try another example
    sample_text2 = "Human:Can you explain why the sky is blue in a way a 5-year-old could understand?\n\nAssistant:"
    print(f"\nTokenizing prompt: '{sample_text2}'")
    inputs = tokenizer(sample_text2, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    print(f"Generating response...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nModel Response: {response}")

except Exception as e:
    print(f"Error during model testing: {e}")
    import traceback
    traceback.print_exc()
    print("\nPlease make sure you've run the previous cell first to load the model.")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Tokenizing prompt: 'Human:Explain how a neural network works like you’re teaching it to someone who’s never coded

Assistant:'
Generating response...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Model Response: Human:Explain how a neural network works like you’re teaching it to someone who’s never coded

Assistant: A neural network is like a giant brain made up of many small parts called neurons. Each neuron has a special job to do, and they work together to solve problems and learn new things. Just like how you can teach someone to do something by showing them step by step, a neural network can be trained by providing it with lots of examples and letting it figure out the patterns on its own. It's kind of like teaching a computer to recognize faces or understand language.


Tokenizing prompt: 'Human:Can you explain why the sky is blue in a way a 5-year-old could understand?

Assistant:'
Generating response...

Model Response: Human:Can you explain why the sky is blue in a way a 5-year-old could understand?

Assistant: Sure! The sky is blue because of the way the sun's light travels through the air. The air is made up of tiny particles called molecules, and these molecules sc